# Methods Lab — build your project's methods notebook

**AHLI Health AI Summer Camp 2026 · Day 4 workshop**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahli-org/ahli-summer-camp-website/blob/main/public/notebooks/methods-lab-template.ipynb)

> **How to use this notebook.** It is a *template*, not a finished analysis. Work
> through the sections on **your own project**: state your project in Section 0,
> build a data substrate (the synthetic generator below, or a public/real
> dataset), and adapt each move to your outcome type and decision. The
> intellectual work is *deciding what to build and how to interpret it* — use an
> AI coding assistant freely to scaffold the synthetic stand-in. Section 7
> converts your findings into your Project Workbook section.

Reuse your Day 3 evaluation moves as the *comparison harness* here: every model goes through the same evaluation.

## Section 0 — Project setup

*Restate your project and reuse your data substrate from Day 3 (the synthetic generator is repeated here so this notebook stands alone).*

In [ ]:
import numpy as np
import pandas as pd

def make_synthetic(
    n=4000,
    prevalence=0.15,        # event rate of the positive class
    signal=1.2,             # strength of the genuine signal
    subgroup_gap=0.6,       # how much weaker the signal is in subgroup B
    shortcut_strength=2.0,  # a feature spuriously predictive in TRAIN only
    label_noise=0.05,       # fraction of labels flipped (uniform)
    shift=True,             # apply a train -> test distribution shift
    seed=0,
):
    """A binary-outcome stand-in with two subgroups, a planted shortcut, label
    noise, and an optional train->test shift. You control the ground truth, so
    you can demonstrate a metric reporting the wrong thing. Adapt the outcome
    type (multiclass / time-to-event / regression) to match YOUR project."""
    rng = np.random.default_rng(seed)
    subgroup = rng.integers(0, 2, n)                       # 0 = A, 1 = B
    x1 = rng.normal(0, 1, n)
    x2 = rng.normal(0, 1, n)
    # Genuine signal, weaker in subgroup B (a true performance gap):
    strength = signal * np.where(subgroup == 1, 1 - subgroup_gap, 1.0)
    logit = strength * x1 + 0.5 * x2 + np.log(prevalence / (1 - prevalence))
    p = 1 / (1 + np.exp(-logit))
    y = rng.binomial(1, p)
    # Label noise:
    flip = rng.random(n) < label_noise
    y = np.where(flip, 1 - y, y)
    # A shortcut: a feature correlated with y in TRAIN, broken at test time.
    split = rng.random(n) < 0.7
    shortcut = np.where(split, shortcut_strength * (y - 0.5), 0) + rng.normal(0, 1, n)
    df = pd.DataFrame({"x1": x1, "x2": x2, "shortcut": shortcut,
                       "subgroup": subgroup, "y": y, "is_train": split})
    train, test = df[df.is_train].copy(), df[~df.is_train].copy()
    if shift:  # shift the test distribution so a fragile model degrades
        test["x1"] = test["x1"] + 0.8
    return train.drop(columns="is_train"), test.drop(columns="is_train")

train_df, test_df = make_synthetic()
FEATURES = ["x1", "x2", "shortcut"]
X_train, y_train = train_df[FEATURES], train_df["y"]
X_test,  y_test  = test_df[FEATURES],  test_df["y"]
print(train_df.shape, test_df.shape, "event rate:", round(y_train.mean(), 3))


## Section 1 — Baseline

A penalized linear model is the bar any complex method must clear. If a heavier model can't beat this, the complexity isn't buying you anything.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def evaluate(model, name):
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    print(f"{name:24s} AUROC {auc:.3f}")
    return auc

evaluate(LogisticRegression(max_iter=1000, C=1.0), "Penalized logistic")

## Section 2 — Gradient-boosted trees

A strong tabular workhorse. Does it beat the baseline on YOUR metric, or just on AUROC?

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
evaluate(HistGradientBoostingClassifier(max_depth=3), "Gradient-boosted trees")

## Section 3 — A small neural network

Capacity is a choice to justify, not a default. (Swap in a torch model if your project needs one.)

In [ ]:
from sklearn.neural_network import MLPClassifier
evaluate(MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500), "Small MLP")

## Section 4 — Representation / foundation-model approach

*When does leveraging a pretrained, general representation help, and when does a focused task-specific model win? For imaging/text/genomics, treat your encoder's output as the feature matrix and slot it in here.* Below is a placeholder representation (PCA) to keep the harness runnable; replace it with your project's representation.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
evaluate(make_pipeline(PCA(n_components=3), LogisticRegression(max_iter=1000)),
         "Representation + linear")

## Section 5 — Comparison harness

Run every model through the **same** evaluation (reuse your Day 3 moves), and look at learning curves vs. sample size. The lesson is usually that complexity often does not help.

In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np

sizes, train_scores, val_scores = learning_curve(
    HistGradientBoostingClassifier(max_depth=3), X_train, y_train,
    train_sizes=np.linspace(0.2, 1.0, 5), scoring="roc_auc", cv=3)
print("sizes:", sizes)
print("val AUROC:", val_scores.mean(axis=1).round(3))
# TODO: plot learning curves; add calibration & net benefit to the harness.

## Section 6 — The shortcut, revisited

Which methods most exploit the spurious feature? Trustworthiness is not the same as raw performance — a model that leans on the shortcut scores well in-distribution and fails under shift.

In [ ]:
for feats, label in [(["x1", "x2", "shortcut"], "with shortcut"),
                      (["x1", "x2"], "without shortcut")]:
    m = HistGradientBoostingClassifier(max_depth=3).fit(X_train[feats], y_train)
    auc = roc_auc_score(y_test, m.predict_proba(X_test[feats])[:, 1])
    print(f"{label:18s} AUROC {auc:.3f}")

## Section 7 — Findings → Workbook Part 4

*Convert what you found into your method rationale (Project Workbook Part 4):*

- **Chosen approach** and why it fits your data and your Day 3 metrics.
- **Foundation vs. task-specific** — your decision and reasoning.
- **Fine-tune / prompt / train-from-scratch** — your decision.
- **Baseline** — and why it is or isn't enough.
- **Internal-validation plan.**
- **Main methodological risk**, and a credible alternative.